# Validate the Integration and Registry Pipeline

Load and submit the existing six-step pipeline: merge, transform, train, predict,
compare, and registry registration. The cluster, its managed-identity client ID,
and the default datastore are discovered from the selected workspace. Only the
external registry name comes from this suite's `.env`.

The final `register_job` uploads the mounted MLflow model directly to the
registry using the **compute cluster UAMI**, not the notebook caller or workspace
identity. It requires **AzureML Registry User** for that identity and private
connectivity to the registry and its backing Storage/ACR. Network Connection
Approver alone does not grant asset permissions. No role is granted here.

Executing the submission cell always starts the full pipeline, including
non-cacheable registry registration. The registry assigns an immutable version,
the step verifies it, and `register_metadata/registration.json` records its ID
and version. Native sharing of existing workspace assets is a separate flow
in foundation 08.


In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential
from dotenv import dotenv_values
import requests
from azure.core.credentials import TokenCredential
from azure.ai.ml.entities import AmlCompute
import pandas as pd
from azure.ai.ml import MLClient, load_job
from azure.ai.ml.entities import ManagedIdentityConfiguration

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
for parent in (search_start, *search_start.parents):
    candidate = parent if parent.name == "workspace_readiness" else parent / "notebooks/workspace_readiness"
    if (candidate / ".env.example").is_file():
        SUITE_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from this repository or notebooks/workspace_readiness.")

SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"

env_path = SUITE_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f"Configure {env_path} with your four Azure resource values.")
env = dotenv_values(env_path, encoding="utf-8-sig", interpolate=False)
SUBSCRIPTION_ID = (env["AZURE_SUBSCRIPTION_ID"] or "").strip()
RESOURCE_GROUP = (env["AZURE_RESOURCE_GROUP"] or "").strip()
WORKSPACE_NAME = (env["AZUREML_WORKSPACE_NAME"] or "").strip()
REGISTRY_NAME = (env["AZUREML_REGISTRY_NAME"] or "").strip()

if len(env) != 4 or not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Fill in only the four workspace and registry values in .env.")
if any(value.startswith(("<", "${")) for value in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Replace the .env placeholders with your Azure resource values.")

credential = AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; registry: {REGISTRY_NAME}")

ASSET_NAMES = {
    "data": "readiness-taxi-data",
    "environment": "readiness-taxi-environment",
    "model": "readiness-taxi-model",
}

def arm_get(credential: TokenCredential, resource_id: str, api_version: str) -> dict:
    response = requests.get(
        f"https://management.azure.com{resource_id}",
        params={"api-version": api_version},
        headers={"Authorization": f"Bearer {credential.get_token('https://management.azure.com/.default').token}"},
        timeout=60,
    )
    response.raise_for_status()
    document = response.json()
    if not isinstance(document, dict):
        raise ValueError(f"Expected an ARM resource object for {resource_id}.")
    return document

def resolve_cluster(
    client: MLClient,
    credential: TokenCredential,
    compute_name: str | None = None,
) -> tuple[str, str]:
    if compute_name:
        compute = client.compute.get(compute_name)
    else:
        clusters = [item for item in client.compute.list() if isinstance(item, AmlCompute)]
        if len(clusters) != 1:
            raise ValueError(
                f"Expected one compute cluster, found {len(clusters)}. "
                "Set COMPUTE_NAME explicitly in this notebook's configuration cell."
            )
        name = clusters[0].name
        if not name:
            raise ValueError("The compute cluster has no name.")
        compute = client.compute.get(name)

    if not isinstance(compute, AmlCompute) or not compute.name:
        raise ValueError("COMPUTE_NAME must select a named AML compute cluster.")
    identities = compute.identity.user_assigned_identities if compute.identity else []
    if not identities or len(identities) != 1:
        raise ValueError(f"Cluster {compute.name} must have exactly one user-assigned managed identity.")
    identity = identities[0]
    client_id = identity.client_id
    if not client_id and identity.resource_id:
        resource = arm_get(credential, identity.resource_id, "2023-01-31")
        client_id = resource.get("properties", {}).get("clientId")
    if not isinstance(client_id, str) or not client_id:
        raise ValueError(f"Unable to resolve the client ID for cluster {compute.name}'s identity.")
    return compute.name, client_id

COMPUTE_NAME = None  # Set a name here only when the workspace has multiple clusters.
COMPUTE_NAME, COMPUTE_IDENTITY_CLIENT_ID = resolve_cluster(ml_client, credential, COMPUTE_NAME)
OUTPUT_DATASTORE = ml_client.datastores.get_default().name

pipeline_path = SAMPLE_ROOT / "pipelines/integration-compare-pipeline.yaml"


In [ ]:
job = load_job(
    pipeline_path,
    params_override=[
        {"inputs.automl_compute": COMPUTE_NAME},
        {"inputs.registry": REGISTRY_NAME},
    ],
)
job.settings.default_compute = COMPUTE_NAME
job.settings.default_datastore = OUTPUT_DATASTORE
job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
for child_job in job.jobs.values():
    child_job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
job.display_name = "Readiness taxi integration compare"
job.tags = {"validation": "workspace-readiness", "operation": "integration-compare"}

assert job.inputs["automl_compute"].result() == COMPUTE_NAME
assert job.inputs["registry"].result() == REGISTRY_NAME
assert job.settings.default_compute == COMPUTE_NAME
assert job.settings.default_datastore == OUTPUT_DATASTORE
assert job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
assert all(
    child_job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
    for child_job in job.jobs.values()
)
print(f"Loaded: {type(job).__name__}")
print(f"Compute: {COMPUTE_NAME}")
print(f"Datastore: {OUTPUT_DATASTORE}")
print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")
print(f"External registry: {REGISTRY_NAME}")

submitted_job = ml_client.jobs.create_or_update(job)
print(f"Submitted: {submitted_job.name}")
ml_client.jobs.stream(submitted_job.name)
final_job = ml_client.jobs.get(submitted_job.name)
child_jobs = list(ml_client.jobs.list(parent_job_name=submitted_job.name))
child_summary = pd.DataFrame(
    [
        {
            "name": child.name,
            "display_name": child.display_name,
            "status": child.status,
        }
        for child in child_jobs
    ]
)
display(child_summary)
if final_job.status != "Completed":
    raise RuntimeError(f"Pipeline ended with status {final_job.status}")
if any(child.status != "Completed" for child in child_jobs):
    raise RuntimeError("At least one child job did not complete")
print({name: output.path for name, output in final_job.outputs.items()})


## Expected Result

The parent pipeline and all six child jobs complete, producing trained-model, prediction, comparison, and verified external-registry registration metadata outputs.

Next: `../03_h2o_reference/01_create_reference_binary_model.ipynb`.